## INIT

In [12]:
# ── Imports ───────────────────────────────────────────────────────────────────
import glob
import json
import datetime
import statistics
from uuid import uuid4

import yaml
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, HnswConfigDiff
from fastembed import TextEmbedding
from chonkie import RecursiveChunker
import chonkie

from src.evaluator import Evaluator
from src.all_functionality import async_chat_wrapper

from sentence_transformers import CrossEncoder
import asyncio
import numpy as np

Evaluation functions defined.
Visualization function defined.


2026-03-18 10:38:06 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-03-18 10:38:11 | WARNING  | tensorflow | From d:\DevTools\Python313\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



### Full notion

In [13]:
from pathlib import Path

docs_dir = Path("./rag_exp/notion_docs")
md_paths = sorted(docs_dir.glob("**/*.md"))

notion_docs = {}
for p in md_paths:
    with p.open("r", encoding="utf-8") as fh:
        notion_docs[p.stem] = fh.read()

print(f"Loaded {len(notion_docs)} markdown files from {docs_dir}")


Loaded 6 markdown files from rag_exp\notion_docs


In [14]:
'''notion_docs = {
    key: value for key, value in notion_docs.items() if "create" not in key
}
print(len(notion_docs))'''

'notion_docs = {\n    key: value for key, value in notion_docs.items() if "create" not in key\n}\nprint(len(notion_docs))'

In [15]:
testing_doc = notion_docs["block_reference"] 

In [16]:
testing_doc = "\n\n\n".join(notion_docs.values())

# Experiment

In [6]:
import re
from chonkie import MarkdownChef, CodeChunker, RecursiveChunker

# Method 1: MarkdownChef
print("=" * 60)
print("Method 1: MarkdownChef")
print("=" * 60)
md_chef = MarkdownChef()
md_doc = md_chef.parse(testing_doc)
print(f"Document type: {type(md_doc)}")
print(f"Total chunks: {len(md_doc.chunks)}")
if md_doc.chunks:
    print(f"\nFirst chunk:\n{md_doc.chunks[0]}\n")

# Method 2: CodeChunker
print("=" * 60)
print("Method 2: CodeChunker")
print("=" * 60)
code_chunker = CodeChunker()
print(f"CodeChunker methods: {[m for m in dir(code_chunker) if not m.startswith('_')]}")
code_chunks = code_chunker.chunk(testing_doc)
print(f"Total chunks: {len(code_chunks)}")
if code_chunks:
    print(f"\nFirst chunk:\n{code_chunks[0]}\n")

# Method 3: RecursiveChunker
print("=" * 60)
print("Method 3: RecursiveChunker")
print("=" * 60)
recursive_chunker = RecursiveChunker()
recursive_chunks = recursive_chunker.chunk(testing_doc)
print(f"Total chunks: {len(recursive_chunks)}")
if recursive_chunks:
    print(f"\nFirst chunk:\n{recursive_chunks[0]}\n")


Method 1: MarkdownChef
Document type: <class 'chonkie.types.markdown.MarkdownDocument'>
Total chunks: 73

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation Index
> Fetch the complete documentation index at: https://developers.notion.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Block

> A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).

For example, the following block object represents a `Heading 2` in the Notion U

d:\DevTools\Python313\Lib\site-packages\chonkie\chunker\code.py:76: UserWarning: The language is set to `auto`. This would adversely affect the performance of the chunker. Consider setting the `language` parameter to a specific language to improve performance.
  warnings.warn(


Total chunks: 102

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation Index
> Fetch the complete documentation index at: https://developers.notion.com/llms.txt
> Use this file to discover all available pages before exploring further.



Method 3: RecursiveChunker
Total chunks: 272

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation 

In [7]:
def chunk_by_tags(document: str) -> list:
    # Chunk document by matching opening/closing XML-like tags (e.g. <Note>...</Note>)
    pattern = re.compile(r'<(?P<tag>[A-Za-z0-9_\-]+)[^>]*>.*?</(?P=tag)>', re.DOTALL)

    tag_chunks = []
    start = 0
    for m in pattern.finditer(document):
        if m.start() > start:
            tag_chunks.append({"tag": None, "text": document[start:m.start()]})
        tag_chunks.append({"tag": m.group("tag"), "text": m.group(0)})
        start = m.end()
    if start < len(document):
        tag_chunks.append({"tag": None, "text": document[start:]})

    # remove empty chunks
    tag_chunks = [c for c in tag_chunks if c["text"].strip()]
    return tag_chunks

tag_chunks = chunk_by_tags(testing_doc)

print(f"Total tag-based chunks: {len(tag_chunks)}")
for i, c in enumerate(tag_chunks[:6]):
    snippet = c["text"][:200].replace("\n", "\\n")
    print(f"{i}: tag={c['tag']!r}, len={len(c['text'])}, preview={snippet!r}")

Total tag-based chunks: 121
0: tag=None, len=885, preview='---\\nsource_url: https://developers.notion.com/reference/block.md\\ncategory: major\\ndescription: A block object represents a piece of content within Notion. The API translates the headings, toggles, para'
1: tag='CodeGroup', len=1084, preview='<CodeGroup>\\n  ```json Example Block Object expandable theme={null}\\n  {\\n  \t"object": "block",\\n  \t"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",\\n  \t"parent": {\\n  \t\t"type": "page_id",\\n  \t\t"page_id": "5983'
2: tag=None, len=123, preview='\\n\\nUse the [Retrieve block children](/reference/get-block-children) endpoint to list all of the blocks on a page.\\n\\n## Keys\\n\\n'
3: tag='Note', len=277, preview='<Note>\\n  Fields marked with an \\* are available to integrations with any capabilities. Other properties require read content capabilities in order to be returned from the Notion API. Consult the [inte'
4: tag=None, len=27649, preview='\\n\\n| Field              | Typ

In [8]:
# Global tracking variables for inspection
global_stats = {
    "total_code": 0,
    "total_images": 0,
    "total_tables": 0,
    "total_metadata": 0,
}

def process_tag_chunks(tag_chunks: list) -> list:
    """
    Process tag_chunks through MarkdownChef to extract nested chunks, tables, code, and images.
    Removes auxiliary fields like 'chunk_type' and 'source' from the produced chunks.
    """
    global global_stats

    processed_chunks = []
    md_chef = MarkdownChef()

    for tag_chunk in tag_chunks:
        original_tag = tag_chunk["tag"]
        text = tag_chunk["text"]

        if original_tag is not None:
            processed_chunks.append({
                "tag": original_tag,
                "text": text,
            })
            continue

        # Parse with MarkdownChef
        try:
            md_doc = md_chef.parse(text)
        except Exception:
            # If parsing fails, keep as single text chunk
            print(f"MarkdownChef parsing failed for chunk with tag={original_tag!r}, treating as plain text.")
            processed_chunks.append({
                "tag": original_tag,
                "text": text,
            })
            continue

        # Extract and track metadata
        if md_doc.metadata:
            global_stats["total_metadata"] += len(md_doc.metadata) if isinstance(md_doc.metadata, list) else 1

        # Process parsed chunks from MarkdownChef
        for chunk in md_doc.chunks:
            processed_chunks.append({
                "tag": "text",
                "text": chunk.text,
            })

        # Extract and track tables
        if md_doc.tables:
            global_stats["total_tables"] += len(md_doc.tables)
            for i, table in enumerate(md_doc.tables):
                processed_chunks.append({
                    "tag": "table",
                    "text": table.content,  # Convert table object to string
                    "table_index": i
                })

        # Extract and track code
        if md_doc.code:
            global_stats["total_code"] += len(md_doc.code)
            for i, code in enumerate(md_doc.code):
                processed_chunks.append({
                    "tag": 'CodeGroup',
                    "text": code.content,  # Convert code object to string
                    "code_index": i
                })

        # Extract and track images
        if md_doc.images:
            global_stats["total_images"] += len(md_doc.images)
            for i, image in enumerate(md_doc.images):
                processed_chunks.append({
                    "tag": "image",
                    "text": image.content,  # Convert image object to string
                    "image_index": i
                })

    return processed_chunks

# Apply function to tag_chunks
expanded_chunks = process_tag_chunks(tag_chunks)

print(f"Initial tag chunks: {len(tag_chunks)}")
print(f"Expanded chunks after MarkdownChef processing: {len(expanded_chunks)}")
print(f"\nGlobal statistics:")
print(f"  Total code blocks: {global_stats['total_code']}")
print(f"  Total tables: {global_stats['total_tables']}")
print(f"  Total images: {global_stats['total_images']}")
print(f"  Total metadata: {global_stats['total_metadata']}")

print(f"\nFirst 10 expanded chunks:")
for i, chunk in enumerate(expanded_chunks[:10]):
    snippet = chunk["text"][:100].replace("\n", "\\n")
    type_label = chunk['tag']
    print(f"{i}: tag={chunk['tag']!r}, type={type_label!r}, len={len(chunk['text'])}, preview={snippet!r}")

unique_tags = set(c['tag'] for c in expanded_chunks)
print(f"\nUnique tags in expanded chunks: {unique_tags}")

# find None tags
print(f"\nChunks with None tag:")
for i, chunk in enumerate(expanded_chunks):
    if chunk['tag'] is None:
        snippet = chunk["text"][:100].replace("\n", "\\n")
        print(f"{i}: tag={chunk['tag']!r}, len={len(chunk['text'])}, preview={snippet!r}")

Initial tag chunks: 121
Expanded chunks after MarkdownChef processing: 156

Global statistics:
  Total code blocks: 3
  Total tables: 29
  Total images: 0
  Total metadata: 0

First 10 expanded chunks:
0: tag='text', type='text', len=885, preview='---\\nsource_url: https://developers.notion.com/reference/block.md\\ncategory: major\\ndescription: A bloc'
1: tag='CodeGroup', type='CodeGroup', len=1084, preview='<CodeGroup>\\n  ```json Example Block Object expandable theme={null}\\n  {\\n  \t"object": "block",\\n  \t"id"'
2: tag='text', type='text', len=123, preview='\\n\\nUse the [Retrieve block children](/reference/get-block-children) endpoint to list all of the block'
3: tag='Note', type='Note', len=277, preview='<Note>\\n  Fields marked with an \\* are available to integrations with any capabilities. Other propert'
4: tag='text', type='text', len=1103, preview='\\n#### Block types that support child blocks\\n\\nSome block types contain nested blocks. The following b'
5: tag='table', t

### Parsing big yaml, tables, code

In [9]:
for i, c in enumerate(expanded_chunks):
    tag = c.get("tag")
    text = c.get("text", "") or ""
    if len(text) > 30000:
        break

In [10]:
code_chunker = CodeChunker()
r = code_chunker(c['text'][2:])
len(r)

1

In [11]:
c['text'][43:100]

'openapi: 3.1.0\ninfo:\n  title: Notion API\n  version: 1.0.0'

In [12]:
c['text'][-43:]

'uth:\n      type: http\n      scheme: bearer\n'

##### Manual + regex

In [13]:
# find the first "openapi" occurrence, trim everything before its line, then load YAML
text_fragment = c["text"] or ""

m = re.search(r"^\s*openapi\s*:", text_fragment, flags=re.IGNORECASE | re.MULTILINE)
if not m:
    m = re.search(r"openapi", text_fragment, flags=re.IGNORECASE)

if m:
    # start from the beginning of the line that contains the match
    line_start = text_fragment.rfind("\n", 0, m.start()) + 1
    yaml_text = text_fragment[line_start:]
else:
    yaml_text = text_fragment  # fallback: try to load whole text

try:
    parsed_yaml = yaml.load(yaml_text, Loader=yaml.FullLoader)
    print("YAML loaded, top-level keys:", list(parsed_yaml.keys()) if isinstance(parsed_yaml, dict) else type(parsed_yaml))
except Exception as e:
    print("YAML load failed:", e)
    parsed_yaml = None

YAML loaded, top-level keys: ['openapi', 'info', 'servers', 'security', 'tags', 'paths', 'components']


#### By line

In [14]:
import importlib

# Use PyYAML's safe_load from sys.modules (avoid overwriting existing `yaml` variable)
pyyaml = importlib.import_module("yaml")

def split_zero_indent(text: str) -> list:
    """Split text by lines starting at column 0 that contain a colon (top-level YAML keys)."""
    header_re = re.compile(r'^[^\s].*?:')
    lines = text.splitlines(True)
    blocks = []
    current = ""
    for line in lines:
        if header_re.match(line) and current.strip():
            blocks.append(current)
            current = line
        else:
            current += line
    if current:
        blocks.append(current)
    return blocks

# Use existing yaml_text variable from the notebook
blocks = split_zero_indent(yaml_text)

parsed_results = []
for i, blk in enumerate(blocks, start=1):
    try:
        parsed = pyyaml.safe_load(blk)
        parsed_results.append({"idx": i, "text": blk, "parsed": parsed, "error": None})
    except Exception as e:
        parsed_results.append({"idx": i, "text": blk, "parsed": None, "error": str(e)})

total = len(parsed_results)
parsed_count = sum(1 for r in parsed_results if r["parsed"] is not None)
failed_indices = [r["idx"] for r in parsed_results if r["parsed"] is None]

print(f"Blocks: {total}, Parsed: {parsed_count}, Failed: {len(failed_indices)} ({(parsed_count/total)*100:.1f}% rescued)")

if failed_indices:
    print("Failed indices (sample):", failed_indices[:10])
    first_fail = next(r for r in parsed_results if r["parsed"] is None)
    preview = first_fail["text"][:400].replace("\n", "\\n")
    print(f"First failed idx={first_fail['idx']}, error={first_fail['error']}")
    print("Preview:", preview)

# Expose for downstream cells
rescued_blocks = [r for r in parsed_results if r["parsed"] is not None]
flagged_blocks = [r for r in parsed_results if r["parsed"] is None]

Blocks: 7, Parsed: 7, Failed: 0 (100.0% rescued)


#### Parsing jsons

In [15]:
import json

json_blocks = [json.dumps(r["parsed"], indent=2) for r in rescued_blocks]
json_chunker = CodeChunker(language="json")
json_chunks = [json_chunker.chunk(block) for block in json_blocks]
print(f"Total JSON blocks: {len(json_blocks)}")
print(f"Total JSON chunks: {len(json_chunks)}")

Total JSON blocks: 7
Total JSON chunks: 7


In [16]:
from langchain_text_splitters import RecursiveJsonSplitter

json_splitter = RecursiveJsonSplitter(max_chunk_size=1000)

2026-03-15 23:05:36 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-03-15 23:05:45 | WARNING  | tensorflow | From d:\DevTools\Python313\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



In [17]:
# Split per rescued block, then flatten and improved prints
json_chunks_by_block = [json_splitter.split_text(r["parsed"]) for r in rescued_blocks]
total_blocks = len(json_chunks_by_block)
json_chunks = [chunk for chunks in json_chunks_by_block for chunk in chunks]

print(f"Blocks processed: {total_blocks}")
print(f"Total JSON chunks after RecursiveJsonSplitter: {len(json_chunks)}")

for i, chunk in enumerate(json_chunks[:10]):
    preview = str(chunk)[:200].replace("\n", "\\n")
    print(f"  [{i}] preview={preview}")

Blocks processed: 7
Total JSON chunks after RecursiveJsonSplitter: 92
  [0] preview={"openapi": "3.1.0"}
  [1] preview={"info": {"title": "Notion API", "version": "1.0.0", "termsOfService": "https://notion.notion.site/Terms-and-Privacy-28ffdd083dc3473e9c2da6ec011b58ac"}}
  [2] preview={"servers": [{"url": "https://api.notion.com"}]}
  [3] preview={"security": [{"bearerAuth": []}]}
  [4] preview={"tags": [{"name": "Databases", "description": "Database endpoints"}, {"name": "Data sources", "description": "Data source endpoints"}, {"name": "Pages", "description": "Page endpoints"}, {"name": "Bl
  [5] preview={"paths": {"/v1/blocks/{block_id}/children": {"patch": {"tags": ["Blocks"], "summary": "Append block children", "operationId": "patch-block-children", "parameters": [{"name": "block_id", "in": "path",
  [6] preview={"paths": {"/v1/blocks/{block_id}/children": {"patch": {"responses": {"200": {"content": {"application/json": {"schema": {"properties": {"type": {"type": "string", "const":

In [18]:
json_chunks[1]

'{"info": {"title": "Notion API", "version": "1.0.0", "termsOfService": "https://notion.notion.site/Terms-and-Privacy-28ffdd083dc3473e9c2da6ec011b58ac"}}'

In [19]:
json_splitter.split_text(rescued_blocks[0]["parsed"])

['{"openapi": "3.1.0"}']

#### Ruamel

In [20]:
from ruamel.yaml import YAML
from ruamel.yaml.error import YAMLError

# Mini example: tolerant YAML parsing with ruamel.yaml and helpful error location
# Requires: pip install ruamel.yaml

# Use existing variables (c, re) from the notebook
text_fragment = c.get("text", "") or ""
yaml_text = text_fragment  # In practice, you might want to trim to the relevant section as shown before

yaml = YAML()
try:
    parsed = yaml.load(yaml_text)
    print("YAML loaded successfully; top-level type:", type(parsed))
except YAMLError as e:
    print("ruamel.yaml parse error:", e)
    mark = getattr(e, "problem_mark", None) or getattr(e, "context_mark", None)
    if mark:
        ln, col = mark.line, mark.column
        print(f"Error location -> line {ln+1}, column {col+1}")
        lines = yaml_text.splitlines()
        start = max(0, ln - 3)
        end = min(len(lines), ln + 4)
        for i in range(start, end):
            prefix = ">>" if i == ln else "  "
            print(f"{prefix} {i+1}: {lines[i]}")
    else:
        print("No line/column info available from the error.")

ruamel.yaml parse error: while scanning for the next token
found character '`' that cannot start any token
  in "<unicode string>", line 1, column 1:
    `yaml patch /v1/blocks/{block_id ... 
    ^ (line: 1)
Error location -> line 1, column 1
>> 1: `yaml patch /v1/blocks/{block_id}/children
   2: openapi: 3.1.0
   3: info:
   4:   title: Notion API


In [21]:
r = recursive_chunker.chunk(text)
r[0]

Chunk(text='`yaml patch /v1/blocks/{block_id}/children
openapi: 3.1.0
info:
  title: Notion API
  version: 1.0.0
  termsOfService: >-
    https://notion.notion.site/Terms-and-Privacy-28ffdd083dc3473e9c2da6ec011b58ac
servers:
  - url: https://api.notion.com
security:
  - bearerAuth: []
tags:
  - name: Databases
    description: Database endpoints
  - name: Data sources
    description: Data source endpoints
  - name: Pages
    description: Page endpoints
  - name: Blocks
    description: Block endpoints
  - name: Comments
    description: Comment endpoints
  - name: File uploads
    description: File upload endpoints
  - name: OAuth
    description: OAuth endpoints (basic authentication)
  - name: Users
    description: User endpoints
  - name: Search
    description: Search endpoints
paths:
  /v1/blocks/{block_id}/children:
    patch:
      tags:
        - Blocks
      summary: Append block children
      operationId: patch-block-children
      parameters:
        - name: block_id
    

In [22]:
# Use existing chunkers (code_chunker, recursive_chunker). Do not re-import.
CHUNK_SIZE = getattr(recursive_chunker, "chunk_size", 2048)

recursive_chunker = RecursiveChunker(chunk_size=CHUNK_SIZE)
code_chunker = CodeChunker(language="yaml")

code_group_inspect = []
recursive_inspect = []
table_inspect = []
other_inspect = []

for i, c in enumerate(expanded_chunks):
    tag = c.get("tag")
    text = c.get("text", "") or ""
    if len(text) > 5000:
        print(f"Long chunk at idx={i} with tag={tag!r}, len={len(text)}")
        if tag == "CodeGroup":
            code_subs = code_chunker.chunk(text)
            # code_chunker returns strings; keep first few chars for preview
            preview = code_subs[0][:500] if code_subs else ""
            code_group_inspect.append({"idx": i, "n_subchunks": len(code_subs), "preview": preview})
        elif tag == "text" and len(text) > CHUNK_SIZE:
            sub_chunks = recursive_chunker.chunk(text)
            # RecursiveChunker returns Chunk objects with .text
            n = len(sub_chunks)
            first = getattr(sub_chunks[0], "text", str(sub_chunks[0]))[:500] if n else ""
            recursive_inspect.append({"idx": i, "n_subchunks": n, "first_preview": first})
        elif tag == "table":
            rows = [r for r in text.splitlines() if r.strip()]
            table_inspect.append({"idx": i, "n_rows": len(rows), "preview": "\n".join(rows[:5])})
        else:
            other_inspect.append({"idx": i, "tag": tag, "len": len(text), "preview": text[:200]})

# Quick summary prints
print(f"CHUNK_SIZE={CHUNK_SIZE}")
print(f"CodeGroup chunks found: {len(code_group_inspect)}")
for item in code_group_inspect[:5]:
    print(f"  idx={item['idx']}, subchunks={item['n_subchunks']}, preview_len={len(item['preview'])}")

print(f"Long text chunks (split by RecursiveChunker): {len(recursive_inspect)}")
for item in recursive_inspect[:5]:
    print(f"  idx={item['idx']}, subchunks={item['n_subchunks']}, first_preview_len={len(item['first_preview'])}")

print(f"Table chunks: {len(table_inspect)}")
for item in table_inspect[:5]:
    print(f"  idx={item['idx']}, rows={item['n_rows']}, preview_lines=\n{item['preview']}")

print(f"Other chunks sampled: {len(other_inspect)}")
for item in other_inspect[:5]:
    print(f"  idx={item['idx']}, tag={item['tag']}, len={item['len']}, preview={item['preview']!r}")

Long chunk at idx=5 with tag='table', len=26544
Long chunk at idx=137 with tag='CodeGroup', len=154748
Long chunk at idx=138 with tag='CodeGroup', len=81024
Long chunk at idx=144 with tag='CodeGroup', len=149871
Long chunk at idx=148 with tag='table', len=9760
Long chunk at idx=155 with tag='table', len=14608
CHUNK_SIZE=2048
CodeGroup chunks found: 3
  idx=137, subchunks=1, preview_len=500
  idx=138, subchunks=1, preview_len=500
  idx=144, subchunks=1, preview_len=500
Long text chunks (split by RecursiveChunker): 0
Table chunks: 3
  idx=5, rows=14, preview_lines=
| Field              | Type                                                                    | Description                                                                                                                                                                                                                                                                                                                                  

In [28]:
# find a table chunk > 20000 chars and split with RecursiveChunker(chunk_size=1000)
CHUNK_SIZE = 5000

rc = RecursiveChunker(chunk_size=CHUNK_SIZE)

match = next(((i, c) for i, c in enumerate(expanded_chunks)
              if c.get("tag") == "table" and len(c.get("text", "")) > 20000), None)

if match is None:
    print("No table chunk >20000 chars found.")
else:
    idx, chunk = match
    text = chunk.get("text", "")
    print(f"Found table chunk at idx={idx}, len={len(text)}. Splitting with CHUNK_SIZE={CHUNK_SIZE}...")
    parts = rc.chunk(text)
    print(f"Produced {len(parts)} sub-chunks. Showing first 5:")
    for j, p in enumerate(parts[:5]):
        p_text = getattr(p, "text", p)
        preview = p_text.replace("\n", " ").replace(" ", "").replace("-", "")[:100]
        print(f"#{j}: len={len(p_text)}, preview={preview}...")

Found table chunk at idx=5, len=26544. Splitting with CHUNK_SIZE=5000...
Produced 7 sub-chunks. Showing first 5:
#0: len=3792, preview=|Field|Type|Description|Examplevalue||:|:|:|:|...
#1: len=3792, preview=|`object`\*|`string`|Always`"block"`.|`"block"`||`id`\*|`string`(UUIDv4)|Identifierfortheblock.|`"7a...
#2: len=3792, preview=|`parent`|`object`|Informationabouttheblock'sparent.See[Parentobject](/reference/parentobject).|`{"t...
#3: len=3792, preview=|`created_time`|`string`([ISO8601datetime](https://en.wikipedia.org/wiki/ISO_8601))|Dateandtimewhent...
#4: len=3792, preview=|`last_edited_time`|`string`([ISO8601datetime](https://en.wikipedia.org/wiki/ISO_8601))|Dateandtimew...


In [27]:
match

(5,
 {'tag': 'table',
  'text': '| Field              | Type                                                                    | Description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

Unique tags in expanded chunks: {'Check', 'Note', 'Danger', 'text', 'Tip', 'Frame', 'Steps', 'Info', 'Warning', 'table', 'CodeGroup'}


#### Fixing stuff

In [40]:
import importlib
import json
import re

pyyaml = importlib.import_module("yaml")

CHUNK_SIZE = 1024
recursive_chunker_1024 = RecursiveChunker(chunk_size=CHUNK_SIZE)
json_chunker = CodeChunker(language="json")

def _as_text_parts(parts):
    out = []
    for part in parts or []:
        out.append(getattr(part, "text", str(part)))
    return out

def extract_yaml_candidates(text: str) -> list[str]:
    candidates = []

    # Prefer fenced YAML blocks when available.
    fenced = re.findall(r"```(?:yaml|yml)\s*\n(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    for block in fenced:
        if block.strip():
            candidates.append(block.strip())

    # Fallback: capture from first likely YAML key (common in OpenAPI/spec docs).
    start_match = re.search(r"(?im)^\s*(openapi|swagger|info|paths|components|servers)\s*:\s*", text)
    if start_match:
        line_start = text.rfind("\n", 0, start_match.start()) + 1
        tail = text[line_start:].strip()
        if tail:
            candidates.append(tail)

    # Final fallback: if text itself looks YAML-ish, include it.
    if re.search(r"(?m)^\s*[^#\s][^:\n]*:\s*", text):
        stripped = text.strip()
        if stripped:
            candidates.append(stripped)

    # Deduplicate while preserving order.
    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            unique.append(c)
            seen.add(c)
    return unique

def parse_yaml_elements(candidates: list[str]) -> list:
    parsed = []

    def try_parse(fragment: str):
        try:
            obj = pyyaml.safe_load(fragment)
        except Exception:
            return None
        if obj is None:
            return None
        if isinstance(obj, (dict, list)):
            return obj
        return None

    for candidate in candidates:
        # 1) Try whole candidate first.
        obj = try_parse(candidate)
        if obj is not None:
            parsed.append(obj)
            continue

        # 2) Try YAML document splits.
        docs = [d for d in re.split(r"\n---\s*\n", candidate) if d.strip()]
        for doc in docs:
            obj = try_parse(doc)
            if obj is not None:
                parsed.append(obj)

        # 3) Try top-level key blocks (safe salvage mode).
        lines = candidate.splitlines(True)
        blocks = []
        current = ""
        header_re = re.compile(r"^[^\s].*?:")
        for line in lines:
            if header_re.match(line) and current.strip():
                blocks.append(current)
                current = line
            else:
                current += line
        if current.strip():
            blocks.append(current)

        for block in blocks:
            obj = try_parse(block)
            if obj is not None:
                parsed.append(obj)

    return parsed

def split_table_rows(table_text: str) -> list[str]:
    rows = []
    for raw_line in table_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if "|" not in line:
            continue
        # Skip markdown separator rows like |---|---:|
        compact = line.replace("|", "").replace(" ", "")
        if compact and all(ch in "-:" for ch in compact):
            continue
        rows.append(line)
    return rows if rows else [table_text]

def apply_requested_chunking(chunks: list, chunk_size: int = 1024) -> list:
    stage_1 = []

    for chunk in chunks:
        text = (chunk.get("text") or "").strip()
        if not text:
            continue

        tag = chunk.get("tag")

        # 1) Tables: one row = one chunk.
        if tag == "table":
            for row_idx, row_text in enumerate(split_table_rows(text)):
                row_chunk = dict(chunk)
                row_chunk["tag"] = "table_row"
                row_chunk["text"] = row_text
                row_chunk["row_index"] = row_idx
                stage_1.append(row_chunk)
            continue

        # 2) YAML flow: regex extraction -> safe parse salvage -> JSON chunking.
        yaml_candidates = extract_yaml_candidates(text)
        parsed_yaml = parse_yaml_elements(yaml_candidates)

        if parsed_yaml:
            for element_idx, element in enumerate(parsed_yaml):
                json_text = json.dumps(element, ensure_ascii=True, indent=2)
                json_parts = _as_text_parts(json_chunker.chunk(json_text)) or [json_text]
                for json_idx, part_text in enumerate(json_parts):
                    stage_1.append({
                        "tag": "yaml_json",
                        "text": part_text,
                        "source_tag": tag,
                        "yaml_element_index": element_idx,
                        "json_chunk_index": json_idx,
                    })
            continue

        stage_1.append(dict(chunk))

    # 3) Recursive chunker for all chunks with text.
    rc = RecursiveChunker(chunk_size=chunk_size)
    final_chunks = []
    for chunk in stage_1:
        text = chunk.get("text", "") or ""
        if not text:
            continue

        parts = _as_text_parts(rc.chunk(text)) or [text]
        total = len(parts)
        for part_idx, part_text in enumerate(parts):
            segmented = dict(chunk)
            segmented["text"] = part_text
            segmented["segment_index"] = part_idx
            segmented["segment_total"] = total
            final_chunks.append(segmented)

    return final_chunks

fixed_chunks = apply_requested_chunking(expanded_chunks, chunk_size=CHUNK_SIZE)
expanded_chunks = fixed_chunks

print(f"Applied fixes. Total chunks now: {len(expanded_chunks)}")
print(f"table_row chunks: {sum(1 for c in expanded_chunks if c.get('tag') == 'table_row')}")
print(f"yaml_json chunks: {sum(1 for c in expanded_chunks if c.get('tag') == 'yaml_json')}")
print(f"Chunks with segmentation metadata: {sum(1 for c in expanded_chunks if 'segment_index' in c)}")

Applied fixes. Total chunks now: 1709
table_row chunks: 168
yaml_json chunks: 1444
Chunks with segmentation metadata: 1709


### Continue (to summary)

In [34]:
def build_summary_prompt(prior_context, snippet):
    prompt = f"""
You are generating retrieval-oriented technical summaries for markdown chunks.

Context:
<context>{prior_context}</context>

Snippet:
<snippet>{snippet}</snippet>

Instructions:
- Write a compact, factual summary of the snippet content only.
- Use dry technical language (e.g., "implemented", "used", "defines", "returns", "contains").
- Remove narrative filler and meta commentary.
- Do NOT use phrases like:
  - "this snippet is about ..."
  - "the text describes ..."
  - "the section discusses ..."
- Do not repeat the prompt or mention summarization steps.
- Max length: 70 words.
- Preserve high-signal details: field names, API objects, constraints, types, enums, parameters, and behavior.

Type-specific rules:
- Table: report schema/columns and key values or constraints.
- Code: report what is implemented, inputs/outputs, and core logic.
- Image: report concrete visible/embedded technical content only.
- Metadata: report explicit keys and values.
"""
    return prompt.strip()

In [35]:
import random
from src.all_functionality import async_chat_wrapper

# Array of target models to iterate over
MODELS_TO_TEST = ["gemma4", "gemma12", "gemma27"]

async def ask_llm(prompt):
    # Randomly select a model from the list
    selected_model = random.choice(MODELS_TO_TEST)
    
    # Placeholder for LLM interaction
    return await async_chat_wrapper(
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        model_size=selected_model
    )

In [36]:
import asyncio
from typing import Dict, Any

async def summarize_chunks_with_concurrency(chunks: list, max_concurrent: int = 10) -> list:
    """
    For every table/CodeGroup chunk, collect prior context (previous text chunk)
    and ask LLM with summarization prompt. Add "summary" field to these chunks.
    Process in parallel with max_concurrent limit.
    """
    
    # Create a semaphore to limit concurrent requests
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def summarize_single(chunk_idx: int, chunk: Dict[str, Any]) -> None:
        """Summarize a single chunk if it's a table or CodeGroup"""
        if chunk["tag"] not in ["table", "CodeGroup"]:
            return
        
        # Find prior text chunk
        prior_context = ""
        for i in range(chunk_idx - 1, -1, -1):
            if chunks[i]["tag"] == "text":
                prior_context = chunks[i]["text"]
                break
        
        if not prior_context:
            prior_context = "(No prior text context found)"
        
        # Build prompt
        prompt = build_summary_prompt(prior_context, chunk["text"])
        
        # Call LLM with semaphore
        async with semaphore:
            try:
                response = await ask_llm(prompt)
                chunk["summary"] = response
                print(f"✓ Summarized chunk {chunk_idx} (tag: {chunk['tag']}, len: {len(chunk['text'])})")
            except Exception as e:
                chunk["summary"] = f"Error: {str(e)}"
                print(f"✗ Error summarizing chunk {chunk_idx}: {str(e)}")
    
    # Create tasks for all chunks
    tasks = [summarize_single(i, chunk) for i, chunk in enumerate(chunks)]
    
    # Run all tasks concurrently with semaphore limit
    await asyncio.gather(*tasks)
    
    return chunks

# Run summarization
print("Starting parallel summarization of table/CodeGroup chunks...")
print(f"Processing {len([c for c in expanded_chunks if c['tag'] in ['table', 'CodeGroup']])} chunks with max_concurrent=10")
print()

expanded_chunks = await summarize_chunks_with_concurrency(expanded_chunks, max_concurrent=10)

print("\nSummarization complete!")
print(f"Chunks with summaries: {len([c for c in expanded_chunks if 'summary' in c])}")

# Show samples
print("\nSample summaries:")
for i, chunk in enumerate(expanded_chunks):
    if "summary" in chunk:
        print(f"\n[Chunk {i}] Tag: {chunk['tag']}")
        print(f"Preview: {chunk['text'][:150].replace(chr(10), ' ')}...")
        print(f"Summary: {chunk['summary'][:200]}")
        if i >= 2:  # Show first 3 with summaries
            break


Starting parallel summarization of table/CodeGroup chunks...
Processing 45 chunks with max_concurrent=10



2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 83 (tag: CodeGroup, len: 237)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 73 (tag: CodeGroup, len: 521)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 65 (tag: CodeGroup, len: 260)


2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop
✓ Summarized chunk 95 (tag: CodeGroup, len: 360)
[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop
✓ Summarized chunk 60 (tag: CodeGroup, len: 274)


...

[async_chat_wrapper] Model: gemma-3-12b-it, finish_reason: stop
✓ Summarized chunk 228 (tag: CodeGroup, len: 446)


2026-03-16 00:00:02 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-12b-it, finish_reason: stop
✓ Summarized chunk 240 (tag: CodeGroup, len: 277)


2026-03-16 00:00:31 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-12b-it, finish_reason: stop
✓ Summarized chunk 129 (tag: CodeGroup, len: 400)

Summarization complete!
Chunks with summaries: 105

Sample summaries:

[Chunk 4] Tag: CodeGroup
Preview: <CodeGroup>   ```json Example Block Object expandable theme={null}   {   	"object": "block",   	"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",   	"pare...
Summary: The snippet represents a block object with an ID of `c02fc1d3-db8b-45c5-a222-27595b15aea7`. It’s a child of page ID `59833787-2cf9-4fdf-8782-e53db20768a5`. The block is a heading level 2, containing t


In [44]:
for i in range(len(expanded_chunks)):
    c = expanded_chunks[i]
    if c.get("tag") != "CodeGroup":
        c['summary'] = None

In [46]:
def build_final_tag(chunks):
    new_chunks = []
    for chunk in chunks:
        if chunk.get("summary"):
            name = "summary"
        else:
            name = "text"
            
        chunk["final"] = f"<{chunk['tag']}>{chunk[name]}</{chunk['tag']}>"
        new_chunks.append(chunk)
    return new_chunks

final_chunks = build_final_tag(expanded_chunks)

In [47]:
# Minimal inspections on final_chunks
print("=" * 60)
print("Final Chunks Inspection")
print("=" * 60)

print(f"\nTotal final chunks: {len(final_chunks)}")

# Count chunks by tag
tag_counts = {}
for chunk in final_chunks:
    tag = chunk.get("tag")
    tag_counts[tag] = tag_counts.get(tag, 0) + 1

print(f"\nChunks by tag:")
for tag, count in sorted(tag_counts.items()):
    print(f"  {tag}: {count}")

# Check for summary presence
chunks_with_summary = len([c for c in final_chunks if "summary" in c])
print(f"\nChunks with summaries: {chunks_with_summary}")

# Length statistics
lengths = [len(chunk.get("final", "")) for chunk in final_chunks]
print(f"\nFinal field length statistics:")
print(f"  Min: {min(lengths)}")
print(f"  Max: {max(lengths)}")
print(f"  Mean: {sum(lengths) / len(lengths):.0f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]}")

# Show sample of each tag type
print(f"\nSample chunks by tag:")
seen_tags = set()
for i, chunk in enumerate(final_chunks):
    tag = chunk.get("tag")
    if tag not in seen_tags:
        print(f"\n[{tag}] Chunk {i} (len: {len(chunk['final'])})")
        print(f"  Preview: {chunk['final'][:150].replace(chr(10), ' ')}...")
        seen_tags.add(tag)

Final Chunks Inspection

Total final chunks: 1709

Chunks by tag:
  Check: 1
  CodeGroup: 45
  Danger: 1
  Frame: 1
  Info: 2
  Note: 13
  Steps: 2
  Tip: 1
  table_row: 168
  text: 23
  yaml_json: 1444

Chunks with summaries: 1709

Final field length statistics:
  Min: 29
  Max: 1047
  Mean: 729
  Median: 884

Sample chunks by tag:

[yaml_json] Chunk 0 (len: 389)
  Preview: <yaml_json>{   "source_url": "https://developers.notion.com/reference/block.md",   "category": "major",   "description": "A block object represents a ...

[CodeGroup] Chunk 4 (len: 345)
  Preview: <CodeGroup>The snippet represents a block object with an ID of `c02fc1d3-db8b-45c5-a222-27595b15aea7`. It’s a child of page ID `59833787-2cf9-4fdf-878...

[text] Chunk 6 (len: 136)
  Preview: <text>  Use the [Retrieve block children](/reference/get-block-children) endpoint to list all of the blocks on a page.  ## Keys  </text>...

[Note] Chunk 7 (len: 290)
  Preview: <Note><Note>   Fields marked with an \* are available t

apply recursive chunker for all text chunks with a UPPERCASED local varibles CHUNK_SIZE

## QDRANT CLIENT

#### LOAD COLLECTION

In [17]:
from qdrant_client.models import Distance, HnswConfigDiff, PointStruct
import secrets
import string

# ──────────────────────────────────────────────────────────────────────────────
# 1. Setup Qdrant client
# ──────────────────────────────────────────────────────────────────────────────

qdrant_client = QdrantClient(path="./rag_exp/qdrant_data", prefer_grpc=True)
COLLECTION_NAME = "notion_chunks"

embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

#### new collection

In [18]:
# ──────────────────────────────────────────────────────────────────────────────
# Retrieve all points with vectors from current collection (fixed offset handling)
# ──────────────────────────────────────────────────────────────────────────────

def retrieve_all_points_with_vectors(client, collection_name: str, batch_size: int = 200) -> list:
    """
    Retrieve all points with their vectors from a Qdrant collection.
    Returns a list of PointStruct/Record objects with vectors and payloads intact.
    """
    all_points = []
    offset = 0
    safety_iters = 0
    MAX_ITERS = 10000
    seen_point_ids = set()

    print(f"Retrieving all points from '{collection_name}'...")

    while True:
        # Scroll with with_vectors=True to include embeddings
        res = client.scroll(
            collection_name=collection_name,
            limit=batch_size,
            offset=offset,
            with_payload=True,
            with_vectors=True
        )

        # qdrant_client.scroll can return (points, next_offset) or just points
        if isinstance(res, (list, tuple)) and len(res) >= 1:
            pts = res[0]
            next_offset = res[1] if len(res) > 1 else None
        else:
            pts = res
            next_offset = None

        if not pts:
            print("No more points to retrieve.")
            break

        new_found = False
        for p in pts:
            pid = getattr(p, "id", None) or (p.get("id") if isinstance(p, dict) else None)
            if pid in seen_point_ids:
                continue
            seen_point_ids.add(pid)
            new_found = True
            all_points.append(p)

        if not new_found:
            print("No new unique points in this batch; stopping.")
            break

        # Use next_offset when provided by Qdrant; otherwise advance by number of points fetched
        if next_offset is not None:
            offset = next_offset
        else:
            offset += len(pts)

        safety_iters += 1
        if safety_iters % 10 == 0:
            print(f"  Progress: {len(all_points)} points retrieved...")

        if safety_iters >= MAX_ITERS:
            print("Reached MAX_ITERS; stopping.")
            break

        # If API signalled no continuation token/offset, break
        if next_offset is None and len(pts) < batch_size:
            break

    print(f"✓ Retrieved {len(all_points)} points total")
    return all_points

# Execute retrieval
points_from_collection = retrieve_all_points_with_vectors(qdrant_client, COLLECTION_NAME)
print(f"Ready to migrate {len(points_from_collection)} points with vectors")


Retrieving all points from 'notion_chunks'...
✓ Retrieved 1709 points total
Ready to migrate 1709 points with vectors


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Create new collection with different configuration
# ──────────────────────────────────────────────────────────────────────────────

from qdrant_client.models import VectorParams, HnswConfigDiff, ScalarQuantizationConfig, models

def create_new_collection_with_config(
    client,
    new_collection_name: str,
    vector_dim: int,
    hnsw_config: dict = None,
    quantization_config: dict = None,
    distance: str = "cosine"
) -> None:
    """
    Create a new Qdrant collection with custom configuration.
    
    Args:
        client: QdrantClient instance
        new_collection_name: Name for the new collection
        vector_dim: Vector dimension (should match retrived points)
        hnsw_config: Dict with keys {m, ef_construct, full_scan_threshold}
        quantization_config: Dict with keys {type, quantile, always_ram}
        distance: Distance metric (cosine, l2, dot)
    """
    
    # Validate collection doesn't exist
    if client.collection_exists(new_collection_name):
        raise ValueError(f"Collection '{new_collection_name}' already exists!")
    
    # Use defaults if not specified
    if hnsw_config is None:
        hnsw_config = {
            "m": 16,
            "ef_construct": 200,
            "full_scan_threshold": 1000
        }
    
    if quantization_config is None:
        quantization_config = {
            "type": models.ScalarType.INT8,
            "quantile": 0.99,
            "always_ram": True
        }
    
    # Map distance string to Distance enum
    distance_map = {
        "cosine": Distance.COSINE,
        "l2": Distance.EUCLID,
        "dot": Distance.DOT
    }
    distance_enum = distance_map.get(distance.lower(), Distance.COSINE)
    
    # Create collection
    client.create_collection(
        collection_name=new_collection_name,
        vectors_config=VectorParams(
            size=vector_dim,
            distance=distance_enum
        ),
        hnsw_config=HnswConfigDiff(
            m=hnsw_config.get("m", 16),
            ef_construct=hnsw_config.get("ef_construct", 200),
            full_scan_threshold=hnsw_config.get("full_scan_threshold", 1000)
        ),
        quantization_config=models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(
                type=quantization_config.get("type", models.ScalarType.INT8),
                quantile=quantization_config.get("quantile", 0.99),
                always_ram=quantization_config.get("always_ram", True)
            )
        )
    )
    
    print(f"✓ Created new collection '{new_collection_name}'")
    print(f"  Vector dim: {vector_dim}")
    print(f"  Distance: {distance}")
    print(f"  HNSW config: {hnsw_config}")
    print(f"  Quantization: {quantization_config}")


# Example: Create a new collection with different HNSW parameters
NEW_COLLECTION_NAME = "notion_chunks_exp_v2"
qdrant_client.delete_collection(NEW_COLLECTION_NAME)  # Delete if exists for clean slate (use with caution!)

# Get vector dimension from retrieved points
vector_dim = len(points_from_collection[0].vector) if points_from_collection else 384

# Define experimental configuration
experimental_hnsw = {
    "m": 4,  # Increased connectivity (vs 16)
    "ef_construct": 40,  # Increased construction effort (vs 200)
    "full_scan_threshold": 100  # Increased threshold
}

# Create the new collection with experimental config
create_new_collection_with_config(
    client=qdrant_client,
    new_collection_name=NEW_COLLECTION_NAME,
    vector_dim=vector_dim,
    hnsw_config=experimental_hnsw,
    distance="cosine"
)


✓ Created new collection 'notion_chunks_exp_v2'
  Vector dim: 384
  Distance: cosine
  HNSW config: {'m': 4, 'ef_construct': 40, 'full_scan_threshold': 100}
  Quantization: {'type': <ScalarType.INT8: 'int8'>, 'quantile': 0.99, 'always_ram': True}


In [43]:
# ──────────────────────────────────────────────────────────────────────────────
# Upsert all points to the new collection
# ──────────────────────────────────────────────────────────────────────────────

def upsert_points_to_collection(client, target_collection: str, points: list, batch_size: int = 500) -> None:
    """
    Upsert a list of PointStruct objects to a target collection in batches.
    """
    total_points = len(points)
    
    for i in range(0, total_points, batch_size):
        batch = points[i:i+batch_size]
        client.upsert(
            collection_name=target_collection,
            points=batch
        )
        print(f"  Upserted batch {i//batch_size + 1}: {len(batch)} points")
    
    print(f"✓ Successfully upserted {total_points} points to '{target_collection}'")


# Perform the upsert
print(f"Starting upsert of {len(points_from_collection)} points to '{NEW_COLLECTION_NAME}'...")
upsert_points_to_collection(
    client=qdrant_client,
    target_collection=NEW_COLLECTION_NAME,
    points=points_from_collection,
    batch_size=500
)

# Verify the new collection
collection_info = qdrant_client.get_collection(NEW_COLLECTION_NAME)
print(f"\n✓ Collection '{NEW_COLLECTION_NAME}' verification:")
print(f"  Points count: {collection_info.points_count}")


Starting upsert of 1709 points to 'notion_chunks_exp_v2'...
  Upserted batch 1: 500 points
  Upserted batch 2: 500 points
  Upserted batch 3: 500 points
  Upserted batch 4: 209 points
✓ Successfully upserted 1709 points to 'notion_chunks_exp_v2'

✓ Collection 'notion_chunks_exp_v2' verification:
  Points count: 1709


In [74]:
COLLECTION_NAME = "notion_chunks"

#### Sparse collection

In [26]:
from qdrant_client import models

In [ ]:
from fastembed import SparseTextEmbedding

bm25_name = 'Qdrant/bm25'
splade_name = 'prithivida/Splade_PP_en_v1'
model = SparseTextEmbedding(model_name=splade_name, output_format="dict")

main_query = "How do I create a new page in Notion using the API?"
embedding = list(model.embed(main_query))
print("Sparse embedding dict keys:", embedding)

In [27]:
NEW_COLLECTION_NAME = "notion_chunks_sparse"

sparse_model = SparseTextEmbedding(model_name=bm25_name)

qdrant_client.create_collection(
    NEW_COLLECTION_NAME,
    sparse_vectors_config={
        "sparse_bm25" : models.SparseVectorParams(
        index=models.SparseIndexParams(on_disk=True))
    }
)

sparse_points_from_collection = []
for record in points_from_collection:
    text = record.payload.get("text", "")
    sparse_vector = list(sparse_model.embed(text))[0]
    new_point = models.PointStruct(
        id=record.id,
        payload=record.payload,
        vector={
            "sparse_bm25": models.SparseVector(
                indices=sparse_vector.indices,
                values=sparse_vector.values
            )
        }
    )
    sparse_points_from_collection.append(new_point)

qdrant_client.upsert(
    collection_name=NEW_COLLECTION_NAME,
    points=sparse_points_from_collection
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
main_query = "create page"
embedding = list(sparse_model.embed(main_query))[0]
res = list(qdrant_client.query_points(
    collection_name=NEW_COLLECTION_NAME,
    using="sparse_bm25",
    query=models.SparseVector(
        indices=embedding.indices,
        values=embedding.values
    ),
    limit=10
))[0][1]

for r in res:
    print(f"ID: {r.id}, Score: {r.score}, Payload keys: {list(r.payload.keys())}")
    print(f"Text preview: {r.payload.get('text', '')[:200].replace(chr(10), ' ')}...")

ID: 1681217399, Score: 6.676257133483887, Payload keys: ['chunk_index', 'chunk_id', 'tag', 'text', 'raw_text', 'neighbor_ids']
Text preview: <Note><Note>   **Creating and updating `child_page` blocks**    To create or update `child_page` type blocks, use the [Create a page](/reference/post-page) and the [Update page](/reference/patch-page)...
ID: 844591218, Score: 6.091614246368408, Payload keys: ['chunk_index', 'chunk_id', 'tag', 'text', 'raw_text', 'neighbor_ids']
Text preview: <yaml_json>{   "paths": {     "/v1/pages": {       "post": {         "tags": [           "Pages"         ],         "summary": "Create a page",         "operationId": "post-page",         "parameters"...
ID: 1231705201, Score: 6.077771186828613, Payload keys: ['chunk_index', 'chunk_id', 'tag', 'text', 'raw_text', 'neighbor_ids']
Text preview: <Warning><Warning>   **Some page `properties` are not supported via the API**    A request body that includes `rollup`, `created_by`, `created_time`, `last_edited_by`, or

In [42]:
res

[('points',
  [ScoredPoint(id=845820470, version=0, score=16.395263671875, payload={'chunk_index': 261, 'chunk_id': 'pQ1V2j26', 'tag': 'yaml_json', 'text': '<yaml_json>{\n  "source_url": "https://developers.notion.com/reference/post-page.md",\n  "category": "create_endpoint",\n  "description": "Use this API to create a new [page](/reference/page) as a child of an existing page or [data source](/reference/data-source)."\n}</yaml_json>', 'raw_text': '{\n  "source_url": "https://developers.notion.com/reference/post-page.md",\n  "category": "create_endpoint",\n  "description": "Use this API to create a new [page](/reference/page) as a child of an existing page or [data source](/reference/data-source)."\n}', 'neighbor_ids': ['ReTOYhzs', 'fkIiobI4']}, vector=None, shard_key=None, order_value=None),
   ScoredPoint(id=1231705201, version=0, score=14.11822509765625, payload={'chunk_index': 1108, 'chunk_id': 'iT8GIjTq', 'tag': 'Warning', 'text': '<Warning><Warning>\n  **Some page `properties` ar

#### Hybrid collection

In [ ]:
from qdrant_client import models
from qdrant_client.models import Distance, PointStruct, VectorParams
from fastembed import TextEmbedding, SparseTextEmbedding

HYBRID_COLLECTION_NAME = "notion_chunks_hybrid"
DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse_splade"
SPARSE_MODEL_NAME = "prithivida/Splade_PP_en_v1"
DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Reuse existing models if already initialized in this notebook session.
if "embedding_model" in globals() and embedding_model is not None:
    dense_embedding_model = embedding_model
else:
    dense_embedding_model = TextEmbedding(model_name=DENSE_MODEL_NAME)

if "sparse_hybrid_model" not in globals() or sparse_hybrid_model is None:
    sparse_hybrid_model = SparseTextEmbedding(model_name=SPARSE_MODEL_NAME)

def _to_dense_list(vector_obj):
    if vector_obj is None:
        return None
    if isinstance(vector_obj, dict):
        # If source point has named vectors, prefer the dense key, else take first vector.
        if DENSE_VECTOR_NAME in vector_obj and vector_obj[DENSE_VECTOR_NAME] is not None:
            return list(vector_obj[DENSE_VECTOR_NAME])
        first_value = next(iter(vector_obj.values()), None)
        return list(first_value) if first_value is not None else None
    return list(vector_obj)

def create_hybrid_collection_from_points(
    source_points: list,
    collection_name: str = HYBRID_COLLECTION_NAME,
    recreate: bool = True,
    batch_size: int = 128,
) -> str:
    if not source_points:
        raise ValueError("source_points is empty. Expected points_from_collection from notion_chunks.")

    # Find vector dimension from first valid dense vector.
    vector_dim = None
    for record in source_points:
        dense_candidate = _to_dense_list(getattr(record, "vector", None))
        if dense_candidate:
            vector_dim = len(dense_candidate)
            break
    if vector_dim is None:
        raise ValueError("Could not infer dense vector dimension from source points.")

    if qdrant_client.collection_exists(collection_name):
        if recreate:
            qdrant_client.delete_collection(collection_name=collection_name)
            print(f"Deleted existing collection: {collection_name}")
        else:
            raise ValueError(f"Collection '{collection_name}' already exists. Set recreate=True to rebuild it.")

    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config={
            DENSE_VECTOR_NAME: VectorParams(size=vector_dim, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            SPARSE_VECTOR_NAME: models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=True)
            )
        },
    )
    print(f"Created hybrid collection '{collection_name}' (dense + sparse)")

    upsert_batch = []
    total = len(source_points)

    for i, record in enumerate(source_points, start=1):
        payload = getattr(record, "payload", None) or {}
        record_id = getattr(record, "id", None)
        dense_vector = _to_dense_list(getattr(record, "vector", None))

        text_for_sparse = (payload.get("text") or payload.get("raw_text") or "").strip()
        if not text_for_sparse:
            # Keep deterministic behavior for points with empty text.
            text_for_sparse = f"chunk_id::{payload.get('chunk_id', record_id)}"

        # fastembed returns an iterable; use iter(...) + next(...) for reliability.
        sparse_embedding = next(iter(sparse_hybrid_model.embed(text_for_sparse)))

        # If source record lacks dense vector, backfill from dense model.
        if dense_vector is None:
            dense_vector = list(next(iter(dense_embedding_model.embed(text_for_sparse))))

        hybrid_point = PointStruct(
            id=record_id,
            payload=payload,
            vector={
                DENSE_VECTOR_NAME: dense_vector,
                SPARSE_VECTOR_NAME: models.SparseVector(
                    indices=sparse_embedding.indices,
                    values=sparse_embedding.values,
                ),
            },
        )
        upsert_batch.append(hybrid_point)

        if len(upsert_batch) >= batch_size or i == total:
            qdrant_client.upsert(collection_name=collection_name, points=upsert_batch)
            print(f"Upserted {i}/{total} points")
            upsert_batch = []

    info = qdrant_client.get_collection(collection_name)
    print(f"Hybrid collection ready: {collection_name}")
    print(f"Points count: {info.points_count}")
    return collection_name

def hybrid_query_points(
    main_query: str,
    collection_name: str = HYBRID_COLLECTION_NAME,
    limit: int = 40,
    prefetch_limit: int = 50,
    hnsw_ef: int = 128,
):
    dense_query = list(next(iter(dense_embedding_model.embed(main_query))))
    sparse_query = next(iter(sparse_hybrid_model.embed(main_query)))

    response = qdrant_client.query_points(
        collection_name=collection_name,
        # Run sparse and dense candidate generation in parallel on Qdrant side.
        prefetch=[
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_query.indices,
                    values=sparse_query.values,
                ),
                using=SPARSE_VECTOR_NAME,
                limit=prefetch_limit,
            ),
            models.Prefetch(
                query=dense_query,
                using=DENSE_VECTOR_NAME,
                limit=prefetch_limit,
            ),
        ],
        # Fuse candidate sets with Reciprocal Rank Fusion.
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        search_params=models.SearchParams(hnsw_ef=hnsw_ef),
        limit=limit,
        with_payload=True,
        with_vectors=False,
    )

    return response.points if hasattr(response, "points") else response

# Build / rebuild hybrid collection from existing dense collection points.
hybrid_collection_name = create_hybrid_collection_from_points(
    source_points=points_from_collection,
    collection_name=HYBRID_COLLECTION_NAME,
    recreate=True,
    batch_size=300,
 ) 

# Example hybrid retrieval call.
main_query = "How do I create a new page in Notion using the API?"
hybrid_hits = hybrid_query_points(main_query, collection_name=hybrid_collection_name, limit=40)

print(f"\nHybrid hits: {len(hybrid_hits)}")
for hit in hybrid_hits[:5]:
    payload = getattr(hit, "payload", None) or {}
    preview = (payload.get("text") or payload.get("raw_text") or "")[:180].replace("\n", " ")
    print(f"ID={getattr(hit, 'id', None)} | score={getattr(hit, 'score', None)} | tag={payload.get('tag')} | {preview}")

Deleted existing collection: notion_chunks_hybrid
Created hybrid collection 'notion_chunks_hybrid' (dense + sparse)


#### Create collection

In [ ]:
if qdrant_client.collection_exists(COLLECTION_NAME):
    raise ValueError(f"Collection '{COLLECTION_NAME}' already exists. Please choose a different name or delete the existing collection.")

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
# 2. Utility function: Generate 8-character UUID identifiers
# ──────────────────────────────────────────────────────────────────────────────

def generate_short_uuid(length: int = 8) -> str:
    """Generate a random alphanumeric identifier."""
    chars = string.ascii_letters + string.digits
    return ''.join(secrets.choice(chars) for _ in range(length))

# Create identifier mapping
chunk_ids = {}
for i, chunk in enumerate(expanded_chunks):
    chunk_ids[i] = generate_short_uuid()

print(f"Generated {len(chunk_ids)} unique identifiers")
print(f"Sample IDs: {list(chunk_ids.items())[:5]}")

Generated 1709 unique identifiers
Sample IDs: [(0, 'fQ9sBU8N'), (1, 'xg7JXhNa'), (2, 'XRGvGwYa'), (3, 'CmpfXCb9'), (4, 'oCpa3wZ7')]


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Embed all chunks and prepare points for Qdrant
# ──────────────────────────────────────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────────────────────
# First pass: Collect IDs and embeddings for all chunks
# ──────────────────────────────────────────────────────────────────────────────

chunk_ids = {}
chunk_embeddings = {}

for idx, chunk in enumerate(expanded_chunks):
    chunk_id_str = generate_short_uuid()
    chunk_ids[idx] = chunk_id_str
    
    # Get final field or fall back to text
    embedding_text = chunk.get("final", chunk.get("text", ""))
    
    # Generate embedding
    embedding = list(embedding_model.embed(embedding_text))[0]
    chunk_embeddings[idx] = embedding

print(f"Generated {len(chunk_ids)} unique IDs and embeddings")
print(f"Vector dimension: {len(list(chunk_embeddings.values())[0])}")

# ──────────────────────────────────────────────────────────────────────────────
# Second pass: Build neighbor relationships and create Qdrant points
# ──────────────────────────────────────────────────────────────────────────────

n_neighbors = 1
points = []

for idx, chunk in enumerate(expanded_chunks):
    chunk_id_str = chunk_ids[idx]
    chunk_type = chunk.get("tag", "text")
    raw_text = chunk.get("text", "")
    
    # Build neighbor list (indices N before and after)
    neighbor_uuids = []
    for neighbor_idx in range(max(0, idx - n_neighbors), idx):
        neighbor_uuids.append(chunk_ids[neighbor_idx])
    for neighbor_idx in range(idx + 1, min(len(expanded_chunks), idx + n_neighbors + 1)):
        neighbor_uuids.append(chunk_ids[neighbor_idx])
    
    # Create payload with all chunk info
    payload = {
        "chunk_index": idx,
        "chunk_id": chunk_id_str,
        "tag": chunk_type,
        "text": chunk.get("final"), 
        "raw_text": raw_text,
        "neighbor_ids": neighbor_uuids,
    }
    
    # Create point for Qdrant
    point = PointStruct(
        id=int(chunk_id_str.encode().hex()[:16], 16) % (2**31),  # Convert to numeric ID
        vector=chunk_embeddings[idx],
        payload=payload
    )
    points.append((chunk_id_str, point))

print(f"Built neighbor relationships for {len(points)} chunks")
print(f"Sample point ID: {points[0][0]}, neighbors: {points[0][1].payload['neighbor_ids'][:3]}")


Generated 1709 unique IDs and embeddings
Vector dimension: 384
Built neighbor relationships for 1709 chunks
Sample point ID: wJDk6yBJ, neighbors: ['zjhEs7Yw']


In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
# 5. Create Qdrant collection with HNSW config
# ──────────────────────────────────────────────────────────────────────────────

from qdrant_client.models import VectorParams, HnswConfigDiff, ScalarQuantizationConfig, models

vector_dim = len(points[0][1].vector)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=vector_dim,
        distance=Distance.COSINE
    ),
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=200,
        full_scan_threshold=1000
    ),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,
            quantile=0.99, # Improves accuracy by clipping outliers
            always_ram=True # Keep quantized vectors in RAM for speed
        )
    )
)

print(f"Created collection '{COLLECTION_NAME}' with HNSW config")

# Upload points to Qdrant
points_to_upload = [point for _, point in points]
qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points_to_upload
)

print(f"Uploaded {len(points_to_upload)} points to Qdrant")

# Create mapping from numeric ID back to string ID
id_mapping = {point.id: chunk_id_str for chunk_id_str, point in points}
print(f"ID mapping created: {len(id_mapping)} entries")


Created collection 'notion_chunks' with HNSW config
Uploaded 1709 points to Qdrant
ID mapping created: 1709 entries


### Query

In [ ]:
from typing import Any
import numpy as np
from qdrant_client import QdrantClient
from fastembed import TextEmbedding

# Default to hybrid collection if available
if "HYBRID_COLLECTION_NAME" not in globals():
    HYBRID_COLLECTION_NAME = "notion_chunks_hybrid"

if "qdrant_client" not in globals():
    qdrant_client = QdrantClient(path="./rag_exp/qdrant_data", prefer_grpc=True)
if "embedding_model" not in globals():
    embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

def _normalize_vector(vector: Any) -> list[float] | None:
    if vector is None:
        return None
    if isinstance(vector, dict):
        first_value = next(iter(vector.values()), None)
        return list(first_value) if first_value is not None else None
    return list(vector)

def _numeric_point_id(chunk_id: str) -> int:
    return int(chunk_id.encode().hex()[:16], 16) % (2**31)

def build_all_results(
    main_query: str,
    queries: list[str],
    limit_per_query: int = 5,
    use_neighbours: bool = True,
    collection_name: str = None,
) -> dict[str, Any]:
    if collection_name is None:
        collection_name = HYBRID_COLLECTION_NAME
    
    deduped_queries = []
    seen = set()
    for q in [main_query, *queries]:
        qn = q.strip()
        if not qn:
            continue
        key = qn.lower()
        if key in seen:
            continue
        seen.add(key)
        deduped_queries.append(qn)

    query_plan = [
        {
            "query_id": f"q_{i:02d}",
            "query": q,
        }
        for i, q in enumerate(deduped_queries)
    ]

    return {
        "main_query": main_query,
        "collection_name": collection_name,
        "limit_per_query": limit_per_query,
        "use_neighbours": use_neighbours,
        "query_plan": query_plan,
        "results_by_id": {},
        "ordered_ids": [],
        "query_runs": [],
    }

def _record_to_item(record: Any, source: str, score: float | None) -> dict[str, Any]:
    payload = getattr(record, "payload", None) or {}
    vector = getattr(record, "vector", None)
    chunk_id = payload.get("chunk_id") or str(getattr(record, "id", ""))
    return {
        "id": chunk_id,
        "qdrant_point_id": getattr(record, "id", None),
        "chunk_index": payload.get("chunk_index"),
        "tag": payload.get("tag"),
        "text": payload.get("text"),
        "raw_text": payload.get("raw_text"),
        "neighbor_ids": payload.get("neighbor_ids", []) or [],
        "embedding": _normalize_vector(vector),
        "score": score,
        "source": source,
    }

def _upsert_item(results_by_id: dict[str, Any], ordered_ids: list[str], item: dict[str, Any]) -> None:
    item_id = item["id"]
    existing = results_by_id.get(item_id)
    if existing is None:
        results_by_id[item_id] = item
        ordered_ids.append(item_id)
        return

    # Merge non-empty fields and keep best score when both exist.
    for k, v in item.items():
        if existing.get(k) is None and v is not None:
            existing[k] = v
    if item.get("score") is not None:
        prev = existing.get("score")
        existing["score"] = item["score"] if prev is None else max(prev, item["score"])

def get_results(all_results: dict[str, Any], prefetch_limit: int = 50, hnsw_ef: int = 128) -> dict[str, Any]:
    """
    Execute hybrid (sparse + dense) retrieval using Qdrant's FusionQuery with RRF.
    Replaces standard dense-only query_points with fused retrieval on hybrid collection.
    """
    collection_name = all_results["collection_name"]
    results_by_id = all_results["results_by_id"]
    ordered_ids = all_results["ordered_ids"]
    query_runs = []

    for plan in all_results["query_plan"]:
        query_id = plan["query_id"]
        query_text = plan["query"]

        # Use hybrid_query_points for fused retrieval instead of standard query
        points = hybrid_query_points(
            main_query=query_text,
            collection_name=collection_name,
            limit=all_results["limit_per_query"],
            prefetch_limit=prefetch_limit,
            hnsw_ef=hnsw_ef,
        )

        run_hits = []
        run_neighbor_hits = []

        for p in points:
            item = _record_to_item(p, source="query", score=getattr(p, "score", None))
            _upsert_item(results_by_id, ordered_ids, item)
            run_hits.append(item["id"])

            if not all_results["use_neighbours"]:
                continue

            for neighbor_id in item["neighbor_ids"]:
                try:
                    neighbor_numeric_id = _numeric_point_id(neighbor_id)
                    neighbor_points = qdrant_client.retrieve(
                        collection_name=collection_name,
                        ids=[neighbor_numeric_id],
                        with_payload=True,
                        with_vectors=False,
                    )
                except Exception:
                    continue

                if not neighbor_points:
                    continue

                n_item = _record_to_item(neighbor_points[0], source="neighbor", score=None)
                _upsert_item(results_by_id, ordered_ids, n_item)
                run_neighbor_hits.append(n_item["id"])

        query_runs.append({
            "query_id": query_id,
            "query": query_text,
            "hit_ids": list(dict.fromkeys(run_hits)),
            "neighbor_ids": list(dict.fromkeys(run_neighbor_hits)),
        })

    all_results["query_runs"] = query_runs
    all_results["combined"] = [results_by_id[i] for i in ordered_ids]
    all_results["combined_documents"] = list(all_results["combined"])
    return all_results

#### Translation

In [53]:
from typing import Literal
import json

async def translate_query(
    query: str,
    method: Literal["hyde", "step_back", "decompose"] = "decompose",
    n: int = 5,
) -> list[str]:
    method_guidance = {
        "hyde": "Generate hypothetical answer-oriented retrieval queries with domain terms.",
        "step_back": "Generate more abstract and high-level Notion API queries.",
        "decompose": "Break the task into concrete Notion API sub-queries.",
    }
    guidance = method_guidance[method]

    prompt = f"""
You are a query translator for retrieval-augmented generation.
Given a user query, produce diverse and useful retrieval queries about the Notion API.

User query: {query}
Method: {method}
Method guidance: {guidance}

Requirements:
- Return exactly {n} queries.
- Keep each query concise and technical.
- Include entity names/fields/endpoints when useful.
- Avoid duplicates.
- Don't include 'Notion API' - it's not needed.

Return strict JSON with this schema:
{{"queries": ["...", "..."]}}
""".strip()

    try:
        response = await async_chat_wrapper(
            messages=[{"role": "user", "content": prompt}],
            model_size="gemma4",
            temperature=0.3,
            json_output=True,
        )

        if isinstance(response, dict):
            data = response
        else:
            data = json.loads(response)

        candidates = data.get("queries", []) if isinstance(data, dict) else []
        cleaned = []
        seen = set()
        for q in candidates:
            if not isinstance(q, str):
                continue
            qc = q.strip()
            key = qc.lower()
            if qc and key not in seen:
                seen.add(key)
                cleaned.append(qc)

        if query.strip() and query.strip().lower() not in seen:
            cleaned.insert(0, query.strip())

        return cleaned[:n]
    except Exception as e:
        print(f"Query translation fallback due to error: {e}")
        return [query.strip()] if query.strip() else []

##### experiments

In [ ]:
main_query = "How do I create a new page in Notion using the API?"
queries = await translate_query(main_query, method="decompose", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:02:06 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page in Notion using the API?
  2. Notion API endpoint for creating a new page
  3. Notion API parameters for specifying page properties
  4. Notion API method for creating a new block within a page
  5. Notion API documentation on page creation using the API


In [43]:
main_query = "How do I create a new page?"
queries = await translate_query(main_query, method="decompose", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:03:12 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page?
  2. Notion API endpoint for creating a new page
  3. Notion API method to define page properties
  4. Notion API parameters for specifying page title
  5. Notion API request body structure for creating a page


In [44]:
main_query = "How do I create a new page in Notion?"
queries = await translate_query(main_query, method="decompose", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:03:19 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page in Notion?
  2. Notion API endpoint for creating a new page
  3. Notion API request body format for creating a page
  4. How to specify page properties (title, icon, etc.) using the Notion API
  5. Notion API method for creating a page with a parent block ID


In [42]:
main_query = "How do I create a new page in Notion using the API?"
queries = await translate_query(main_query, method="hyde", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:02:25 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page in Notion using the API?
  2. Notion API createPage endpoint parameters
  3. Notion API page properties data types
  4. Notion API block creation methods
  5. Notion API pagination for page listing


In [ ]:
# 'Notion API'
main_query = "How do I create a new page?"
queries = await translate_query(main_query, method="decompose", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:12:16 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page?
  2. Notion API endpoint for creating a new page
  3. How to specify page properties when creating a new page
  4. Notion API method for adding a title to a new page
  5. Create a new page with a specific parent page ID


In [54]:
# 'Notion API' stricter
main_query = "How do I create a new page?"
queries = await translate_query(main_query, method="decompose", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:13:01 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page?
  2. How do I create a new page using the /pages endpoint?
  3. What is the request body format for creating a new page?
  4. How do I specify the parent page ID when creating a new page?
  5. What properties can be set when creating a new page?


In [70]:
main_query = "How do I create a new page?"
queries = await translate_query(main_query, method="hyde", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:27:52 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page?
  2. Notion page creation endpoint
  3. Notion database page properties
  4. Notion API create block request
  5. Notion page ID format


In [72]:
main_query = "How do I create a new page?"
queries = await translate_query(main_query, method="step_back", n=5)
print("Queries:")
for i, q in enumerate(queries, start=1):
    print(f"  {i}. {q}")

2026-03-17 12:28:09 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
Queries:
  1. How do I create a new page?
  2. Create a new block within a page.
  3. List available page properties.
  4. Retrieve the page creation endpoint.
  5. Implement block insertion via the API.


- "about Notion" but dont' include
- decompoes is much more intelligent
- step back is less relevant
- decompose definitely is teh best

#### Retrieve

In [22]:
# (Helper functions are orchestrated in full_run())

In [23]:
# (Integrated into full_run())

- too much yaml json. It matters tho
- returns reference.md (retrieval maybe) 
    - same with returning parametrs (notion client)
- have duplicate chunks and mmr can't solve it: threshold problems

#### Rerank

In [58]:
# Rerank: accept docs with embedded IDs -> return filtered docs with IDs
from fastembed.rerank.cross_encoder import TextCrossEncoder

def rerank(
    main_query: str,
    docs_with_ids: list[dict],
    cross_name: str = "jinaai/jina-reranker-v1-turbo-en",
    top_k: int = 20,
) -> list[dict]:
    """Rerank docs (with embedded IDs) and return top-k filtered docs with IDs intact."""
    # Filter non-empty docs
    valid_docs = [d for d in docs_with_ids if (d.get("text") or "").strip()]
    if not valid_docs:
        return []

    doc_ids = [d["id"] for d in valid_docs]
    texts = [d["text"] for d in valid_docs]

    try:
        cross_encoder = TextCrossEncoder(model_name=cross_name)
        ranked = list(cross_encoder.rerank(main_query, texts, top_k=min(top_k, len(texts))))

        if ranked and isinstance(ranked[0], dict) and "index" in ranked[0]:
            selected_indices = [r["index"] for r in ranked]
        elif ranked and isinstance(ranked[0], (float, int)):
            scored = list(enumerate(ranked))
            scored.sort(key=lambda x: x[1], reverse=True)
            selected_indices = [i for i, _ in scored[:top_k]]
        else:
            raise ValueError("Unexpected reranker output format")

        return [valid_docs[i] for i in selected_indices]
    except Exception as e:
        print(f"Reranker fallback due to error: {e}")
        return valid_docs[:top_k]

- more k is probably better

#### MMR

In [ ]:
def _cosine_similarity_matrix(query_vector: np.ndarray, doc_vectors: np.ndarray) -> np.ndarray:
    q = query_vector / (np.linalg.norm(query_vector) + 1e-12)
    d = doc_vectors / (np.linalg.norm(doc_vectors, axis=1, keepdims=True) + 1e-12)
    return d @ q

def maximal_marginal_relevance(
    query_vector: np.ndarray,
    docs_with_embeddings: list[dict],
    top_k: int = 5,
    lambda_mult: float = 0.5,
) -> list[dict]:
    """Apply MMR to docs with embedded IDs and embeddings; append 'mmr_score' to each valid doc and return selected docs."""
    if len(docs_with_embeddings) == 0:
        return []

    # keep only docs that have embeddings
    valid_docs = [d for d in docs_with_embeddings if d.get("embedding") is not None]
    if not valid_docs:
        return []

    doc_vectors = np.vstack([np.asarray(d["embedding"], dtype=np.float32) for d in valid_docs])
    doc_to_query = _cosine_similarity_matrix(query_vector, doc_vectors)
    n = len(valid_docs)

    # prepare mmr score storage (None for not-yet-selected / not-selected)
    mmr_scores = [None] * n

    # initial pick: highest relevance
    first_idx = int(np.argmax(doc_to_query))
    selected = [first_idx]
    mmr_scores[first_idx] = float(doc_to_query[first_idx])
    remaining = set(range(n)) - set(selected)

    while len(selected) < min(top_k, n) and remaining:
        best_idx = None
        best_score = -np.inf
        selected_vectors = doc_vectors[selected]

        for idx in list(remaining):
            relevance = float(doc_to_query[idx])
            vec = doc_vectors[idx]
            # redundancy = max cosine similarity between candidate and any selected
            num = selected_vectors @ vec  # shape (k,)
            den = (np.linalg.norm(selected_vectors, axis=1) + 1e-12) * (np.linalg.norm(vec) + 1e-12)
            redundancy = float(np.max(num / den))
            mmr_score = (lambda_mult * relevance) - ((1.0 - lambda_mult) * redundancy)
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx

        if best_idx is None:
            break

        selected.append(best_idx)
        remaining.remove(best_idx)
        mmr_scores[best_idx] = float(best_score)

    # attach mmr_score to each valid doc (None for those not selected)
    for i, doc in enumerate(valid_docs):
        doc["mmr_score"] = mmr_scores[i]

    # return selected docs (with mmr_score present)
    return [valid_docs[i] for i in selected]

# Run MMR
selected = maximal_marginal_relevance(query_vec, demo_docs, top_k=5, lambda_mult=0.6)

# Show results
print("Selected (id, mmr_score):")
for doc in selected:
    print(doc["id"], doc.get("mmr_score"))

# Confirm mmr_score appended to demo_docs
print("\nNumber of demo docs with mmr_score set:", sum(1 for d in demo_docs if d.get("mmr_score") is not None))

Selected (id, mmr_score):
VHRHnSB0 0.9545638561248779
flCh7POG 0.246162462234497
YC39d5Ww 0.10656639337539664
6zors9M4 0.05949367284774776
pQ1V2j26 0.03937450647354124
f8d8KRPG -0.0659683227539063
WdxjMzaL -0.07666382193565367
4bB3OM9o -0.1575178384780884
xp1PullX -0.24239445179700858
TgBinZiF -0.26884074658155444
ReTOYhzs -0.5616732716560364
CvIjs12M -0.5761691451072692
LjUapnmV -0.6799742937088014
rq5MrkDc -0.7021403789520264
A9ixlEMB -0.7209548234939576
ARnTMv00 -0.7473003983497619
fkIiobI4 -0.7773690462112427
fbYX2Wpr -0.8351611614227294
sV5eZfpP -0.8371173381805419
WtyfD98y -0.8546676874160766

Number of demo docs with mmr_score set: 20


- threshold for score? uncalibrated, high cliff, lambda tuning, small llm, docs + set(ids),approx top k

#### Answer

In [ ]:


async def answer_query(main_query: str, docs_with_ids: list[dict], model_size: str = "gemma27") -> str:
    """Generate answer from docs with embedded IDs; extract text for context."""
    cleaned_docs = [d.get("text", "") for d in docs_with_ids if (d.get("text") or "").strip()]
    context = "\n\n".join(cleaned_docs[:8])
    if not context:
        return "No supporting context was retrieved."

    prompt = f"""
You are a precise assistant answering a Notion API question.
Use only the provided context. If context is insufficient, say so briefly.

Question:
{main_query}

Context:
{context}
""".strip()

    try:
        response = await async_chat_wrapper(
            messages=[{"role": "user", "content": prompt}],
            model_size=model_size,
            temperature=0.2,
        )
        return response
    except Exception as e:
        return f"LLM call failed: {e}"

- quite relevant & concise, minimal hallucination

#### NLI

In [ ]:


def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / (np.sum(e) + 1e-12)

def nli_verify(
    generated_answer: str,
    docs_with_ids: list[dict],
    nli_model_name: str = "dleemiller/EttinX-nli-s",
) -> dict[str, dict]:
    """Verify generated answer against docs with embedded IDs; return per-doc NLI labels."""
    cross_encoder = CrossEncoder(nli_model_name)
    texts = [d.get("text", "") for d in docs_with_ids]
    pairs = [(text, generated_answer) for text in texts]
    raw_scores = cross_encoder.predict(pairs)

    results = {}
    labels_3way = ["contradiction", "entailment", "neutral"]

    for doc_dict, s in zip(docs_with_ids, raw_scores):
        doc_id = doc_dict.get("id", "unknown")
        doc_text = doc_dict.get("text", "")
        
        if isinstance(s, (list, tuple, np.ndarray)) and len(s) >= 3:
            arr = np.asarray(s[:3], dtype=np.float32)
            probs = _softmax(arr)
            best_idx = int(np.argmax(probs))
            label = labels_3way[best_idx]
            confidence = float(probs[best_idx])
            raw = arr.tolist()
        else:
            val = float(s)
            label = "entailment" if val >= 0.5 else "contradiction"
            confidence = float(abs(val))
            raw = [val]

        results[doc_id] = {
            "label": label,
            "confidence": confidence,
            "doc_preview": (doc_text or "")[:220],
            "raw": raw,
        }

    return results

- qutie dumb because json (and just raw RAG chunks) doesn't make sense for it -> neutral. Untrustable & unhelpful

#### pipe run

In [ ]:
async def full_run(
    main_query: str,
    limit_per_query: int = 5,
    use_neighbours: bool = True,
    translation_method: Literal["hyde", "step_back", "decompose"] = "decompose",
    rerank_top_k: int = 5,
    mmr_top_k: int = 5,
    answer_model: str = "gemma27",
    nli_model: str = "dleemiller/EttinX-nli-s",
) -> dict[str, Any]:
    """Minimal end-to-end RAG pipeline: translate → retrieve → rerank → mmr → answer → verify."""
    
    # 1. Translate query
    queries = await translate_query(main_query, method=translation_method, n=5)
    
    # 2. Retrieve (using hybrid collection with fused sparse+dense retrieval)
    all_results = build_all_results(
        main_query=main_query,
        queries=queries,
        limit_per_query=limit_per_query,
        use_neighbours=use_neighbours,
        collection_name=HYBRID_COLLECTION_NAME,
    )
    all_results = get_results(all_results)
    
    # 3. Extract retrieved docs
    retrieved_ids = list(all_results["ordered_ids"])
    retrieved_docs_with_ids = [
        {"id": cid, "text": all_results["results_by_id"][cid].get("text") or ""}
        for cid in retrieved_ids
    ]
    
    # 4. Rerank
    reranked_docs_with_ids = rerank(main_query, retrieved_docs_with_ids, top_k=rerank_top_k)
    reranked_ids = [d["id"] for d in reranked_docs_with_ids]
    
    # 5. MMR
    # Compute query vector on-the-fly for MMR (since hybrid_query_points handles embeddings internally)
    query_vec = np.asarray(list(next(iter(dense_embedding_model.embed(main_query)))), dtype=np.float32)
    mmr_input_docs_with_ids = [
        {**d, "embedding": all_results["results_by_id"][d["id"]].get("embedding")}
        for d in reranked_docs_with_ids
    ]
    mmr_docs_with_ids = maximal_marginal_relevance(
        query_vector=query_vec,
        docs_with_embeddings=mmr_input_docs_with_ids,
        top_k=min(mmr_top_k, len(mmr_input_docs_with_ids)),
        lambda_mult=0.6,
    )
    mmr_ids = [d["id"] for d in mmr_docs_with_ids]
    
    # 6. Generate answer
    answer = await answer_query(main_query, mmr_docs_with_ids, model_size=answer_model)
    
    # 7. Verify with NLI
    verification_results = nli_verify(answer, mmr_docs_with_ids, nli_model_name=nli_model)
    
    return {
        "main_query": main_query,
        "answer": answer,
        "mmr_docs_with_ids": mmr_docs_with_ids,
        "mmr_ids": mmr_ids,
        "reranked_ids": reranked_ids,
        "retrieved_ids": retrieved_ids,
        "verification_results": verification_results,
        "all_results": all_results,
    }

In [115]:
main_query = "What properties can be set when creating a new page?"
pipeline_result = await full_run(
    main_query,
    limit_per_query=15, 
    use_neighbours=False,
    rerank_top_k=20,
    mmr_top_k=7,
    translation_method="decompose"
)

print(f"Query: {pipeline_result['main_query']}")
print(f"Answer: {pipeline_result['answer']}\n")
print(f"Retrieved: {len(pipeline_result['retrieved_ids'])} docs")
print(f"Reranked: {len(pipeline_result['reranked_ids'])} docs")
print(f"MMR selected: {len(pipeline_result['mmr_ids'])} docs")
print(f"\nMMR doc IDs: {pipeline_result['mmr_ids']}")

label_counts = {}
for item in pipeline_result['verification_results'].values():
    label = item["label"]
    label_counts[label] = label_counts.get(label, 0) + 1
print(f"\nNLI label distribution: {label_counts}")

2026-03-17 12:58:58 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop


2026-03-17 12:59:10 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


2026-03-17 12:59:12 | INFO     | sentence_transformers.cross_encoder.CrossEncoder | Use pytorch device: cpu


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: What properties can be set when creating a new page?
Answer: When creating a new page, you can set properties within the `properties` object. These properties are defined as key-value pairs where the key is the property name and the value corresponds to a `pagePropertyValueWithIdResponse` schema.

However, the following properties are **not** supported and will cause an error if included: `rollup`, `created_by`, `created_time`, `last_edited_by`, and `last_edited_time`. These are Notion-generated and cannot be set via the API.

Supported properties include: `url`, `public_url`, `parent`, `icon`, `cover`, and `properties` (which itself contains further properties).

An example of setting a "Name" property with a title is:

```yaml
properties: {
  Name: {
    title: [{ text: { content: "New Page Title" } }]
  }
}
```

The `title` property accepts an array of [rich text objects](https://developers.notion.com/reference/rich-text).

Retrieved: 35 docs
Reranked: 20 docs
MMR selected: 7

- hyde into some general unrelated things. And answer included reference md links 
- more docs helps

#### Report

In [116]:
async def evaluate_rag_comprehensiveness(pipeline_result: dict, max_docs: int = 5) -> dict:
    """
    Ask the LLM (gemini-3.1-flash-lite-preview) to evaluate RAG output for
    comprehensiveness w.r.t. the Notion API. Returns a parsed JSON-like dict.

    Uses existing async_chat_wrapper from the notebook.
    """
    main_query = pipeline_result.get("main_query", "")
    answer = pipeline_result.get("answer", "")
    mmr_docs = pipeline_result.get("mmr_docs_with_ids", [])[:max_docs]

    docs_snippets = []
    for d in mmr_docs:
        did = d.get("id", "<no-id>")
        txt = (d.get("text") or "")[:800].replace("\n", " ")
        docs_snippets.append(f"{did}: {txt}")

    prompt = f"""
You are an expert technical auditor for retrieval-augmented-generation (RAG).
Evaluate the comprehensiveness and fidelity of the provided RAG answer + top documents against standalone Notion API docs.

Inputs:
- Query: {main_query}
- Generated answer: {answer}
- Retrieved top documents (id: snippet):\n{chr(10).join(docs_snippets)}

Tasks (return STRICT JSON, no extra text):
1) overall_score: integer 0-10 (comprehensiveness & correctness).
2) coverage: list of concrete Notion API topics/fields that the answer covers.
3) missing: list of important Notion API topics/fields absent from the answer.
4) hallucinations: list of specific claims in the answer not supported by the provided docs.
5) recommendations: short actionable steps to improve retrieval or prompt (max 6 items).

Constraints:
- Be brief and precise in lists.
- Use only evidence visible in the snippets; when unsure, mark as 'uncertain'.
- The parts such asmissing, coverage, and others can be empty.
"""
    # call LLM
    resp = await async_chat_wrapper(
        messages=[{"role": "user", "content": prompt}],
        model_size="gemini-3.1-flash-lite-preview",
        temperature=0.0,
        json_output=True,
    )

    # Normalize returned JSON/dict
    try:
        if isinstance(resp, dict):
            return resp
        else:
            return json.loads(resp)
    except Exception:
        return {"error": "LLM response parse failed", "raw": resp}

# Example usage:
result = await evaluate_rag_comprehensiveness(pipeline_result)
print(json.dumps(result, indent=2, ensure_ascii=False))

2026-03-17 12:59:15 | WARNING  | src.all_functionality | Model size 'gemini-3.1-flash-lite-preview' not found in map, using as-is: gemini-3.1-flash-lite-preview
2026-03-17 12:59:17 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemini-3.1-flash-lite-preview, finish_reason: stop
{
  "overall_score": 6,
  "coverage": [
    "properties object structure",
    "unsupported properties (rollup, created_by, created_time, last_edited_by, last_edited_time)",
    "title property format",
    "parent property",
    "icon and cover properties"
  ],
  "missing": [
    "icon property schema details",
    "distinction between page creation request body vs response schema",
    "required fields for page creation (e.g., parent)",
    "locked property"
  ],
  "hallucinations": [
    "Claiming 'url' and 'public_url' are supported properties to set when creating a new page (these are read-only response fields in the provided docs)",
    "Claiming 'icon' is a supported property to set (not explicitly supported by provided snippets)"
  ],
  "recommendations": [
    "Clarify that 'url' and 'public_url' are response fields, not input parameters for creation.",
    "Retrieve documentation specifically for t

- hallucinates hallucination
    - request for empty didn't help
    - too harsh
- uncalibrated at all. More info !-> better score

In [117]:
def build_pipeline_report(pipeline_result: dict) -> dict:
    """Extract and structure results from unified pipeline execution."""
    all_results = pipeline_result["all_results"]
    nli_results = pipeline_result["verification_results"]
    
    retrieved_count = len(all_results.get("ordered_ids", []))
    with_embeddings = sum(
        1 for cid in all_results.get("ordered_ids", [])
        if all_results["results_by_id"][cid].get("embedding") is not None
    )

    label_counts = {}
    for item in nli_results.values():
        label = item.get("label", "unknown")
        label_counts[label] = label_counts.get(label, 0) + 1

    return {
        "main_query": pipeline_result["main_query"],
        "answer": pipeline_result["answer"],
        "retrieval_count": retrieved_count,
        "embeddings_count": with_embeddings,
        "reranked_ids": pipeline_result["reranked_ids"],
        "mmr_ids": pipeline_result["mmr_ids"],
        "nli_labels": label_counts,
    }

# Generate report from pipeline results
pipeline_report = build_pipeline_report(pipeline_result)

print("=== PIPELINE RESULTS ===")
print(f"Query: {pipeline_report['main_query']}")
print(f"Retrieved: {pipeline_report['retrieval_count']} docs ")
print(f"With embeddings: {pipeline_report['embeddings_count']}")
print(f"Reranked: {len(pipeline_report['reranked_ids'])} docs")
print(f"MMR selected: {len(pipeline_report['mmr_ids'])} docs")
print(f"NLI labels: {pipeline_report['nli_labels']}")
print(f"\nAnswer:\n{pipeline_report['answer'][:800]}")

=== PIPELINE RESULTS ===
Query: What properties can be set when creating a new page?
Retrieved: 35 docs 
With embeddings: 35
Reranked: 20 docs
MMR selected: 7 docs
NLI labels: {'neutral': 7}

Answer:
When creating a new page, you can set properties within the `properties` object. These properties are defined as key-value pairs where the key is the property name and the value corresponds to a `pagePropertyValueWithIdResponse` schema.

However, the following properties are **not** supported and will cause an error if included: `rollup`, `created_by`, `created_time`, `last_edited_by`, and `last_edited_time`. These are Notion-generated and cannot be set via the API.

Supported properties include: `url`, `public_url`, `parent`, `icon`, `cover`, and `properties` (which itself contains further properties).

An example of setting a "Name" property with a title is:

```yaml
properties: {
  Name: {
    title: [{ text: { content: "New Page Title" } }]
  }
}
```

The `title` property accepts an ar

In [118]:
# Expanded pipeline inspection (top-3 previews, truncated to 300 chars)
def _preview(text, n=300):
    return (text or "")[:n].replace("\n", " ").strip()

results_by_id = pipeline_result.get("all_results", {}).get("results_by_id", {})

# If pipeline_result contains only id lists, map them to dicts with text from results_by_id.
retrieved_ids_list = pipeline_result.get("retrieved_ids", []) or []
retrieved_docs_with_ids = [
    {"id": cid, "text": (results_by_id.get(cid) or {}).get("text") or ""}
    for cid in retrieved_ids_list
]

reranked_ids_list = pipeline_result.get("reranked_ids", []) or []
reranked_docs_with_ids = [
    {"id": cid, "text": (results_by_id.get(cid) or {}).get("text") or ""}
    for cid in reranked_ids_list
]

print(f"Query: {pipeline_result.get('main_query')}")
print("\nAnswer (truncated 1000 chars):")
print(pipeline_result.get('answer', ''))
print()

print(f"Retrieved: {len(retrieved_docs_with_ids)} docs")
print(f"Reranked: {len(reranked_docs_with_ids)} docs")
print(f"MMR selected: {len(pipeline_result.get('mmr_ids', []))} docs")
print()

# All queries
print("All queries in plan:")
for plan in pipeline_result.get("all_results", {}).get("query_plan", []):
    print(f"  - {plan.get('query_id')}: {plan.get('query')}")
print()

# Top-3 Retrieved
'''print("Top 3 Retrieved docs (id: preview 300 chars):")
for doc in retrieved_docs_with_ids[:3]:
    print(f"  - {doc['id']}: {_preview(doc.get('text'), 300)}")
print()'''

# Top-3 Reranked
print("Top 3 Reranked docs (id: preview 300 chars):")
for doc in reranked_docs_with_ids[:30]:
    print(f"  - {doc['id']}: {_preview(doc.get('text'), 300)}")
print()

# Top-3 MMR
'''print("Top 3 MMR docs (id: preview 300 chars):")
for doc in pipeline_result.get('mmr_docs_with_ids', [])[:3]:
    print(f"  - {doc['id']}: {_preview(doc.get('text'), 300)}")
print()'''

# NLI distribution
verification = pipeline_result.get('verification_results', {})
label_counts = {}
for v in verification.values():
    label_counts[v.get('label', 'unknown')] = label_counts.get(v.get('label', 'unknown'), 0) + 1
print("NLI label distribution:", label_counts)
print()

# NLI details for top-3 MMR docs
print("NLI details for top-3 MMR docs:")
for doc in pipeline_result.get('mmr_docs_with_ids', [])[:15]:
    vid = doc.get('id')
    nli = verification.get(vid)
    if nli:
        print(f"  - {vid}: label={nli.get('label')}, confidence={nli.get('confidence'):.3f}, doc_preview={_preview(nli.get('doc_preview'),200)}")
    else:
        print(f"  - {vid}: (no NLI result)")

# Pipeline report summary (if available)
if 'pipeline_report' in globals():
    print()
    print("Pipeline report summary:")
    print(f"  main_query: {pipeline_report.get('main_query')}")
    print(f"  retrieval_count: {pipeline_report.get('retrieval_count')}")
    print(f"  embeddings_count: {pipeline_report.get('embeddings_count')}")
    print(f"  reranked_ids (sample): {pipeline_report.get('reranked_ids')[:5]}")
    print(f"  mmr_ids (sample): {pipeline_report.get('mmr_ids')[:5]}")
    print(f"  nli_labels: {pipeline_report.get('nli_labels')}")

Query: What properties can be set when creating a new page?

Answer (truncated 1000 chars):
When creating a new page, you can set properties within the `properties` object. These properties are defined as key-value pairs where the key is the property name and the value corresponds to a `pagePropertyValueWithIdResponse` schema.

However, the following properties are **not** supported and will cause an error if included: `rollup`, `created_by`, `created_time`, `last_edited_by`, and `last_edited_time`. These are Notion-generated and cannot be set via the API.

Supported properties include: `url`, `public_url`, `parent`, `icon`, `cover`, and `properties` (which itself contains further properties).

An example of setting a "Name" property with a title is:

```yaml
properties: {
  Name: {
    title: [{ text: { content: "New Page Title" } }]
  }
}
```

The `title` property accepts an array of [rich text objects](https://developers.notion.com/reference/rich-text).

Retrieved: 35 docs
Reranked:

In [119]:
for doc in pipeline_result.get('mmr_docs_with_ids', [])[:15]:
    print(f"Doc ID: {doc.get('id')}")
    print(f"Text snippet: {_preview(doc.get('text'), 5000)}")
    nli = pipeline_result.get('verification_results', {}).get(doc.get('id'))
    if nli:
        print(f"NLI label: {nli.get('label')}, confidence: {nli.get('confidence'):.3f}")
    else:
        print("No NLI result for this doc.")
    print("-" * 80)

Doc ID: iT8GIjTq
Text snippet: <Warning><Warning>   **Some page `properties` are not supported via the API**    A request body that includes `rollup`, `created_by`, `created_time`, `last_edited_by`, or `last_edited_time` values in the properties object returns an error. These Notion-generated values cannot be created or updated via the API. If the `parent` contains any of these properties, then the new page’s corresponding values are automatically created. </Warning></Warning>
NLI label: neutral, confidence: 0.675
--------------------------------------------------------------------------------
Doc ID: QIQqVJCK
Text snippet: <yaml_json>"properties": {           "type": {             "type": "string",             "const": "link_to_page"           },           "link_to_page": {             "anyOf": [               {                 "title": "Page Id",                 "type": "object",                 "properties": {                   "type": {                     "type": "string",        

## analysis

In [ ]:
raise

In [14]:
# Query Qdrant for CodeGroup payloads and print them (fixed infinite loop + dedupe)
MAX_TO_PRINT = globals().get("MAX_TO_PRINT", 5)
collection = COLLECTION_NAME
code_groups = []
offset = 0
batch = globals().get("batch", 200)

seen_point_ids = set()
safety_iters = 0
MAX_ITERS = 10000

while True:
    pts = qdrant_client.scroll(collection_name=collection, limit=batch, offset=offset, with_payload=True)[0]
    if not pts:
        break

    new_found = False
    for p in pts:
        if p.id in seen_point_ids:
            continue
        seen_point_ids.add(p.id)
        payload = p.payload or {}
        if payload.get("tag") == "CodeGroup":
            code_groups.append((payload.get("chunk_index"), payload, p.id))
        new_found = True

    # safety: if no new points in this batch, stop to avoid infinite loop
    if not new_found:
        print("No new unique points returned in this batch; stopping to avoid infinite loop.")
        break

    offset += len(pts)
    safety_iters += 1
    if safety_iters >= MAX_ITERS:
        print("Reached maximum iteration safety limit; aborting scroll.")
        break

print(f"Found {len(code_groups)} CodeGroup chunks in collection '{collection}'.")
if not code_groups:
    print("No CodeGroup chunks available.")
else:
    shown = 0
    for idx, payload, point_id in code_groups:
        raw_text = payload.get("raw_text") or payload.get("text") or ""
        summary = payload.get("text") or "(No summary available)"

        print("\n" + "=" * 100)
        print(f"Chunk index: {idx}, point id: {point_id}")
        print("Tag:", payload.get("tag"))
        print("\n[CODE GROUP TEXT]")
        print(raw_text)
        print("\n[SUMMARY]")
        print(summary)

        shown += 1
        if shown >= MAX_TO_PRINT:
            break

    if shown == 0:
        print("CodeGroup chunks found, but none has a non-None summary.")

No new unique points returned in this batch; stopping to avoid infinite loop.
Found 6 CodeGroup chunks in collection 'notion_chunks'.

Chunk index: 240, point id: 843800952
Tag: CodeGroup

[CODE GROUP TEXT]
<CodeGroup>
  ```json Example Unsupported block object theme={null}
  {
    "object": "block",
    "id": "7af38973-3787-41b3-bd75-0ed3a1edfac9",
    "type": "unsupported",
    //...other keys excluded
    "unsupported": {
      "block_type": "form"
    }
  }
  ```
</CodeGroup>

[SUMMARY]
<CodeGroup>The JSON example defines an "unsupported" block object with `type` set to `"unsupported"`. It contains an `unsupported` object, which includes the `block_type` field. The `block_type` value identifies the internal block type as a string, such as `"form"`.</CodeGroup>

Chunk index: 132, point id: 844002925
Tag: CodeGroup

[CODE GROUP TEXT]
<CodeGroup>
  ```json Example Image block object theme={null}
  {
    // ... other keys excluded
    "type": "image",
    // ... other keys excluded
   

In [13]:
code_groups

[(240,
  {'chunk_index': 240,
   'chunk_id': 'd85V2Kax',
   'tag': 'CodeGroup',
   'text': '<CodeGroup>The JSON example defines an "unsupported" block object with `type` set to `"unsupported"`. It contains an `unsupported` object, which includes the `block_type` field. The `block_type` value identifies the internal block type as a string, such as `"form"`.</CodeGroup>',
   'raw_text': '<CodeGroup>\n  ```json Example Unsupported block object theme={null}\n  {\n    "object": "block",\n    "id": "7af38973-3787-41b3-bd75-0ed3a1edfac9",\n    "type": "unsupported",\n    //...other keys excluded\n    "unsupported": {\n      "block_type": "form"\n    }\n  }\n  ```\n</CodeGroup>',
   'neighbor_ids': ['DAXm3t9C', 'MzRLrqSB']},
  843800952),
 (132,
  {'chunk_index': 132,
   'chunk_id': 'tIu62Nvm',
   'tag': 'CodeGroup',
   'text': '<CodeGroup>An Image block object contains a `type` key with value "image". It includes an `image` key, which defines an `external` type with a `url` field pointing to 